# eGenes plotting 
- all genes - snp+indel - CC/SN combined
- panel A and B

In [ ]:
import pandas as pd
import torch
import tensorqtl
from tensorqtl import pgen, cis, trans, post

import io
import sys
import post 
import glob
import re

import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib
import os

from statsmodels.stats.multitest import multipletests
import statsmodels

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"torch: {torch.__version__} (CUDA {torch.version.cuda}), device: {device}")
print(f"pandas: {pd.__version__}")
print(f"tensorqtl: {tensorqtl.__version__}")


matplotlib.rcParams['pdf.fonttype'] = 42

In [ ]:
# cell type name correction
cell_type_labels_corr={"L2_3_IT": "L2/3 IT","L5_IT": "L5 IT", "L6_CT": "L6 CT", "L4_IT": "L4 IT", "L5_IT_A": "L5 IT A", "L5_IT_B": "L5 IT B", "L6_IT": "L6 IT", "L6_IT_Car3": "L6 IT Car3", "L5_6_NP": "L5/6 NP", "neurons" : "Neurons"}


In [ ]:
def calculate_and_extract_qvalues(cis_df, fdr=0.05, qvalue_lambda=0.85):
    output_buffer = io.StringIO()
    sys.stdout = output_buffer
    post.calculate_qvalues(cis_df, fdr=fdr, qvalue_lambda=qvalue_lambda)
    sys.stdout = sys.__stdout__
    output = output_buffer.getvalue()

    # Print the captured output for debugging
    print("Captured Output:")
    print(output)

    # Extract the min p-value threshold for the given FDR
    fdr_str = f"{fdr:.2f}"
   
    min_p_value_threshold_match = re.search(rf'min p-value threshold @ FDR {fdr_str}:\s*([\d\.e-]+)', output)
    if min_p_value_threshold_match:
        min_p_value_threshold = float(min_p_value_threshold_match.group(1))
    else:
        min_p_value_threshold = float('nan') 
    print(f"Extracted min p-value threshold @ FDR {fdr}: {min_p_value_threshold}")
    column_name = f'min_p-value_threshold_at_FDR_{fdr}'
    cis_df[column_name] = min_p_value_threshold

    return cis_df

In [ ]:
def process_directory(path, directory, prefix_list, fdr):

    cis_dfs = []  # To store cis DataFrames


    for prefix in prefix_list:
        prefix_res = f'eQTL_{prefix}'
        cis_df = pd.read_csv(f'{path}/results/{directory}/{prefix_res}.csv.gz', index_col=0)  # output from map_cis
        cell_type = prefix.split('_')[2:]
        cell_type = "_".join(cell_type)
        cis_df = calculate_and_extract_qvalues(cis_df, fdr=fdr)
        cis_df = cis_df[cis_df['qval'] <= fdr]
        cis_df["cell_type"] = cell_type
        cis_df["variant_type"] = directory
        cis_dfs.append(cis_df)


    return cis_dfs 

In [ ]:
def process_all_directories(path, directories, prefix_list, fdr):
    all_egenes = pd.DataFrame()
    all_cis_dfs = []


    for directory in directories:
        cis_dfs = process_directory(path, directory, prefix_list, fdr)
        all_cis_dfs.extend(cis_dfs)
     
    combined_cis_df = pd.concat(all_cis_dfs, ignore_index=False)
  
    return combined_cis_df 

## Substantia nigra eGenes

In [ ]:
directories = ['snp', 'indel']

date= 'XXXXX' # change as appropriate

prefix_list = prefix_list = [f'{date}_sn_Astro',
    f'{date}_sn_Micro-PVM',
    f'{date}_sn_OPC',
    f'{date}_sn_Oligo',
    f'{date}_sn_neurons',]

fdr = 0.05
path = '../../snRNA/01_analysis/04_QTL/sn/'
# Process all directories
combined_cis_df_sn = process_all_directories(path, directories, prefix_list, fdr)



In [ ]:
combined_cis_df_sn = combined_cis_df_sn.reset_index()
combined_cis_df_sn['cell_type'] = combined_cis_df_sn['cell_type'].replace(cell_type_labels_corr)
combined_cis_df_sn.groupby(by=['cell_type','variant_type'])['phenotype_id'].nunique()

In [ ]:
combined_cis_df_sn.groupby(by='variant_type').variant_id.nunique()

## Cingulate gyrus eGenes

In [ ]:
directories = ['snp', 'indel']

prefix_list = prefix_list = [
    f'{date}_cc_Astro',
f'{date}_cc_L2_3_IT',
f'{date}_cc_L4_IT',
f'{date}_cc_L5_6_NP',
f'{date}_cc_L5_IT_A',
f'{date}_cc_L5_IT_B',
f'{date}_cc_L6_CT',
f'{date}_cc_L6_IT',
f'{date}_cc_L6b',
f'{date}_cc_Lamp5',
f'{date}_cc_Micro-PVM',
f'{date}_cc_OPC',
f'{date}_cc_Oligo',
f'{date}_cc_Pvalb',
f'{date}_cc_Sncg',
f'{date}_cc_Sst',
f'{date}_cc_Vip',
]

fdr = 0.05
path = '../../snRNA/01_analysis/04_QTL/cc/ct/'
# Process all directories
combined_cis_df_cc = process_all_directories(path, directories, prefix_list, fdr)
combined_cis_df_cc = combined_cis_df_cc.reset_index()
combined_cis_df_cc

In [ ]:
combined_cis_df_cc['cell_type'] = combined_cis_df_cc['cell_type'].replace(cell_type_labels_corr)

In [ ]:
combined_cis_df_cc.groupby(by=['cell_type','variant_type'])['phenotype_id'].nunique()

In [ ]:
combined_cis_df_cc.groupby(by=['cell_type'])['phenotype_id'].nunique()

## All eGenes together

In [ ]:
combined_cis_df_cc['region'] = 'CC'
combined_cis_df_sn['region'] = 'SN'

In [ ]:
combined_cis_df = pd.concat([combined_cis_df_cc,combined_cis_df_sn])

In [ ]:
combined_cis_df.phenotype_id.nunique()

In [ ]:
combined_cis_df['ct_reg'] = combined_cis_df['cell_type'] + ', ' + combined_cis_df['region']

In [ ]:
# barplot with shared eGenes as separate category
shared_dict = {}
for cell_type, group in combined_cis_df.groupby('ct_reg'):
    sets = [set(group[group['variant_type'] == vt]['phenotype_id']) for vt in group['variant_type'].unique()]
    if len(sets) > 1:
        shared_dict[cell_type] = set.intersection(*sets)
    else:
        shared_dict[cell_type] = set()

grouped = (
    combined_cis_df
    .groupby(['ct_reg', 'variant_type'])['phenotype_id']
    .nunique()
    .reset_index(name='count')
)


shared_rows = []
for cell_type, shared_ids in shared_dict.items():
    shared_count = len(shared_ids)
    if shared_count > 0:
        shared_rows.append({'ct_reg': cell_type, 'variant_type': 'shared', 'count': shared_count})


for idx, row in grouped.iterrows():
    cell_type = row['ct_reg']
    variant_type = row['variant_type']
    shared_count = len(shared_dict[cell_type])
    if shared_count > 0:

        only_variant_count = (
            len(set(combined_cis_df[
                (combined_cis_df['ct_reg'] == cell_type) &
                (combined_cis_df['variant_type'] == variant_type)
            ]['phenotype_id']) - shared_dict[cell_type])
        )
        grouped.at[idx, 'count'] = only_variant_count


plot_df = pd.concat([grouped, pd.DataFrame(shared_rows)], ignore_index=True)
pivot_df = plot_df.pivot_table(index='ct_reg', columns='variant_type', values='count', fill_value=0)
pivot_df['total'] = pivot_df.sum(axis=1)
pivot_df = pivot_df.sort_values(by='total', ascending=False)
pivot_df = pivot_df.drop(columns='total')


custom_colors = {
    'snp': '#A5BFCC',
    'indel': '#ffc759',
    'shared': '#F7A5A5', 
}
colors = [custom_colors.get(col, '#333333') for col in pivot_df.columns]

pivot_df.plot(
    kind='bar',
    stacked=True,
    figsize=(10, 4),
    color=colors,
    edgecolor='black'
)
plt.ylabel('Number of eGenes')
plt.xlabel('')
plt.title('Number of significant eGenes')
plt.legend(title='Variant Type')
plt.tight_layout()
plt.xticks(rotation=45, ha='right')
plt.savefig('figures/manuscript_barplot_egenes.pdf', dpi=300, bbox_inches='tight')
plt.show()

## nr cells vs nr eGenes

In [ ]:
cc_nr = pd.read_csv('../../snRNA/01_analysis/04_QTL/cell_numbers/cc_number_of_cells.tsv', sep = '\t')
cc_nr = cc_nr.rename(columns={"consensus_cell_type_corrected": "cell_type"})
sn_nr = pd.read_csv('../../snRNA/01_analysis/04_QTL/cell_numbers/sn_number_of_cells.tsv', sep = '\t')
sn_nr = sn_nr.rename(columns={"cell_type_simplified": "cell_type"})
sn_nr['region'] = 'SN'
cc_nr['region'] = 'CC'

In [ ]:
ct_nr = pd.concat([cc_nr,sn_nr])
ct_nr['cell_type'] = ct_nr['cell_type'].replace(cell_type_labels_corr)
ct_nr['ct_reg'] = ct_nr['cell_type'] + ', ' + ct_nr['region']
ct_nr

In [ ]:
df = combined_cis_df.groupby(by='ct_reg').phenotype_id.nunique()
df

In [ ]:
grouped = ct_nr.join(df, on = 'ct_reg',how ='inner')
grouped

In [ ]:
dict_colors =  {'L2/3 IT': '#ff0000',
                 'Oligo': '#149900',
 'Astro': '#8f00b3',
 'OPC': '#2EF0F0',
 'Micro-PVM': '#bf9360',
 'L5 IT': '#B2F7B7',
 'Vip': '#79baf2',
 'Pvalb': '#FA9A6E',
 'Sst': '#990000',
 'L6 IT': '#F7A400',
 'Lamp5': '#2d4459',
 'L4 IT': '#307cbf',
 'L6 CT': '#4c0000',
 'Sncg': '#FA7ADC',
 'L6b': '#00f220',
 'L5/6 NP': '#accbe6',
 'L5 ET': '#520066',
 'L6 IT Car3': '#330000',
 'VLMC': '#594f43',
 'Endo': '#16591f',
 'Sst Chodl': '#00FF00',
 'Neurons': '#999545',
 'DopaN': '#cc3333',
 'neuron_GLUL+': '#e59900',
 'GabaN': '#ace6b4',
 'OLG/OPC': '#262d33',
 'neuron_unk': '#ee00ff',
 'GlutaN': '#e57373',
 'unk': '#8c5e00',
'immune':'deepskyblue',
 'L5 IT B':'#B2F7B7',
'L5 IT A':'palegreen',
}

In [ ]:
grouped['color'] = grouped['cell_type'].map(dict_colors)

plt.figure(figsize=(8, 6))
scatter_plot = sns.scatterplot(
    data=grouped,
    x='count',
    y='phenotype_id',
    hue='cell_type',  # Color by cell type
    style='region',   # Shape by region
    palette=dict_colors,  # Color palette
    s=100,
    markers={'CC': 'o', 'SN': '^'}
)


sns.regplot(
    data=grouped,
    x='count',
    y='phenotype_id',
    scatter=False,  
    color='grey',   # Line color
    line_kws={'linewidth': 1, 'linestyle': '--'},
    ci = None
)

plt.xscale('log')

# Add labels and title
plt.grid(color='lightgrey', linestyle='--', linewidth=0.5)
plt.xlabel('Number of cells')
plt.ylabel('Number of significant eGenes')
plt.title('Number of significant eGenes variants per cell type (FDR < 0.05)')
plt.legend(title='Cell type', bbox_to_anchor=(1.05, 1), loc='upper left')  # Adjust legend position
plt.tight_layout()

plt.savefig('figures/scatter_eGene_number_cells_log.pdf', format='pdf', dpi=300, bbox_inches='tight')# Show the plot

plt.show()